# 04 · 여러 특징을 함께 쓰면 얼마나 나아지는가

Cat28을 주 모델로 고정하고 Cat18, XGBoost28, 선형·spline 가산 로지스틱, 03의 단일 지표 모델과 비교한다. 트리 모델은 seed 42·43·44를 모두 학습한 뒤 **검증 AP**가 가장 높은 seed를 고른다. 시험 집합은 선택 후 한 번만 평가한다.

**흐름:** 특징·분할 확인 → 모델 정의와 작은 예시 → 11개 후보 학습 → 검증 선택 → 같은 시험 산모의 성능 차이 → 개발 집합 교차검증 → 산출물 저장

**입력:** `data/results/features.csv`, `protocol.json`, 03의 `pred_single_*.csv`  
**출력:** `data/results/`의 후보·선택·예측·시험 비교·교차검증·상태 파일

원본 `03-요약된-파일/04-조합-모델.ipynb`의 모델 설정, seed 선택, 산모 bootstrap과 개발 교차검증을 유지한다. 입력 코호트와 산모 분할은 이 폴더의 00→01→02 결과를 따른다. 이전 폴더의 pilot 결과 숫자를 이 실행의 결과로 사용하지 않는다.

## 1. 입력과 산모 분할을 확인한다

한 행은 특징 추출에 성공한 태아의 5분 신호 구간이다. `target`은 그 구간의 전문가 이상 라벨이다. 같은 산모의 모든 구간은 01에서 정한 하나의 `split`을 공유해야 한다. Cat28의 앞 18개가 Cat18이다.

In [ ]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, PrecisionRecallDisplay
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import SplineTransformer, StandardScaler

RESULTS = Path("data/results")
SEEDS = (42, 43, 44)
TREE_ITERATIONS = 1000
TREE_PATIENCE = 80
THREADS = 2
N_BOOTSTRAP = 2000
CV_FOLDS = 5
BOOTSTRAP_SEED = 42

### 출력에 남기는 오류 정보

오류는 고정된 분류와 건수로만 요약한다. `other_error`의 상세 원인은 내부 오류 파일에서 확인한다. 건수는 오류 내역 수이며, 같은 기록이 여러 번 포함될 수 있다.

In [ ]:
def issue_category(reason):
    # Match known phrases, but return ONLY fixed labels: never echo an exception.
    # Example: "FileNotFoundError: /private/DEMO.json" -> "missing_input".
    text = str(reason)
    groups = {
        "missing_input": ("FileNotFoundError", "No such file", "[Errno 2]"),
        "missing_dependency": ("No module named", "cannot import name", "ImportError"),
        "fhr_missing": ("FHR 결측률",),
        "invalid_wave": ("FHR과 TOCO", "측정 간격", "FHR은 0 이상", "신호 길이"),
        "invalid_bbox": ("Bbox",),
        "invalid_id": ("ID가", "산모 ID", "산모 목록", "행 정보 불일치", "구간 정보 불일치"),
        "invalid_labels": ("라벨이 올바르지", "라벨 수", "라벨·점수"),
        "no_usable_data": ("구간이 없습니다", "비었", "남은 산모가 없습니다"),
        "split_unavailable": ("분할", "양성·음성", "정상·비정상", "least populated class", "test_size"),
        "checkpoint_mismatch": ("체크포인트", "CNN 입력·설정"),
        "io_error": ("PermissionError", "Permission denied", "No space left", "OSError"),
        "invalid_value": ("ValueError", "TypeError", "유한", "could not convert", "ufunc"),
    }
    for category, phrases in groups.items():
        if any(phrase in text for phrase in phrases):
            return category
    return "other_error"

In [ ]:
def show_issue_counts(reasons):
    # Count issue entries, not unique people; one record may have several entries.
    categories = pd.Series([issue_category(reason) for reason in reasons], dtype="string")
    counts = categories.value_counts().rename_axis("category").to_frame("count")
    print("제외·오류 내역 수 =", len(categories))
    # Full messages, IDs, paths, and jobs stay in the internal CSV files.
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        display(counts)

In [ ]:
# Keep the exact feature order from 02 and 03.
CAT28 = [
    "fhr_mean", "fhr_min", "fhr_max", "fhr_sd", "stv", "brady_frac", "tachy_frac",
    "n_decel", "decel_max_depth", "decel_time_frac", "n_accel",
    "toco_mean", "toco_max", "toco_sd", "n_contractions",
    "ft_corr0", "ft_corr_min", "ft_corr_min_lag_s", "figo_baseline", "figo_baseline_var",
    "n_early_decel", "n_late_decel", "n_variable_decel",
    "n_severe_decel", "n_prolonged_decel", "hist_width", "hist_median", "hist_mode",
]
# Example: Cat18 stops at ft_corr_min_lag_s; Cat28 adds ten more features.
CAT18 = CAT28[:18]
ARMS = {
    "cat28": CAT28,
    "cat18": CAT18,
    "xgb28": CAT28,
    "logistic28": CAT28,
    "additive28": CAT28,
}
# Every probability keeps its original row, fetus record, and mother ID.
PREDICTION_COLUMNS = [
    "row_id", "record_id", "mother_id", "site", "target", "split",
    "record_all_normal", "signal_index", "score", "threshold",
]
# Example: 3 tree families × 3 seeds + 2 logistic arms = 11 candidates.
print("features =", len(CAT28), "Cat18 =", len(CAT18), "candidates =", 11)

In [ ]:
feature_table = pd.DataFrame()
protocol = {}
input_error = None

try:
    # IDs are strings; "00DEMO527183" is a fictional example with leading zeros.
    id_types = {key: str for key in ("row_id", "record_id", "mother_id", "site")}
    # One row = one retained 5-minute segment, not one mother.
    feature_table = pd.read_csv(RESULTS / "features.csv", dtype=id_types)
    protocol = json.loads((RESULTS / "protocol.json").read_text(encoding="utf-8"))
except (OSError, ValueError) as error:
    input_error = str(error)

In [ ]:
ready = False
training = pd.DataFrame()
validation = pd.DataFrame()
testing = pd.DataFrame()
issues = []

if input_error is None:
    try:
        if feature_table.empty or not protocol.get("split_ready", False):
            raise ValueError("특징 표가 비었거나 산모 분할이 준비되지 않았습니다")
        # Example: missing hist_width stops this complete five-arm comparison.
        feature_table[CAT28]  # Every model needs its declared feature columns.
        if not feature_table["row_id"].is_unique:
            raise ValueError("구간 ID가 중복됩니다")
        # Example: two records from mother A must both be in train, not train/test.
        if feature_table.groupby("mother_id")["split"].nunique().max() > 1:
            raise ValueError("같은 산모가 여러 분할에 배정되어 있습니다")
        if set(feature_table["split"]) != {"train", "val", "test"}:
            raise ValueError("train/val/test 외의 분할이 있거나 분할이 비었습니다")

        # Reuse 01's saved mother split; no model gets a new holdout.
        training = feature_table.loc[feature_table["split"].eq("train")].copy()
        validation = feature_table.loc[feature_table["split"].eq("val")].copy()
        testing = feature_table.loc[feature_table["split"].eq("test")].copy()
        for part in (training, validation, testing):
            # AUROC and AP need both normal (0) and abnormal (1) windows.
            if set(part["target"].unique()) != {0, 1}:
                raise ValueError("train/val/test 각각 정상·비정상 라벨이 필요합니다")
        ready = True
    except (KeyError, ValueError) as error:
        input_error = str(error)
        issues.append(dict(stage="input", job="features", reason=input_error))
else:
    issues.append(dict(stage="input", job="features", reason=input_error))

In [ ]:
print("ready =", ready)
if ready:
    # Rows count windows; nunique counts mothers with any retained window.
    display(feature_table.groupby("split").agg(
        mothers=("mother_id", "nunique"),
        windows=("row_id", "size"),
        abnormal_fraction=("target", "mean"),
    ))
else:
    print('조합 모델 미산출:', '[details omitted from notebook output]')

## 2. 세 모델 구조를 정의한다

CatBoost28과 CatBoost18은 depth 6, XGBoost28은 depth 4다. 세 계열 모두 최대 1,000회, 학습률 0.05, 검증 조기중단 80회를 쓴다. 선형 가산 모델은 28개 특징을 그대로, spline 가산 모델은 각 특징의 기저값을 함께 쓰되 특징 간 곱 항은 넣지 않는다. 결측 대체, spline knot, 표준화는 **학습 부분에서만** 적합한다.

In [ ]:
class SplineBasis(TransformerMixin, BaseEstimator):
    def fit(self, values, y=None):
        # Example: [140, 140] has range 0 -> constant column.
        # Each of the 28 input columns gets its own varying/constant flag.
        self.keep_ = np.ptp(values, axis=0) > 0
        self.spline_ = None
        if self.keep_.any():
            # Only varying train columns set the 5 uniform knots.
            # Example: range 110–170 -> knots at 110, 125, 140, 155, 170.
            self.spline_ = SplineTransformer(
                n_knots=5, degree=3, knots="uniform",
                extrapolation="constant", include_bias=False,
            )
            self.spline_.fit(values[:, self.keep_])
        return self

    def transform(self, values):
        if self.spline_ is None:
            # Example: 4 rows of constant features -> one zero column per row.
            return np.zeros((len(values), 1))
        # Nonconstant columns each contribute 6 cubic-spline basis values.
        # Example: 2 varying features -> 12 columns, with no cross-products.
        return self.spline_.transform(values[:, self.keep_])

In [ ]:
def make_logistic_model(spline=False):
    # Example: train values [110, 130, NaN, 170] -> NaN becomes median 130.
    steps = [SimpleImputer(strategy="median", keep_empty_features=True)]
    if spline:
        # Additive spline = transform each feature separately, then sum effects.
        steps.append(SplineBasis())
    steps.append(StandardScaler())
    steps.append(LogisticRegression(C=1, max_iter=2000, random_state=42))
    return make_pipeline(*steps)

In [ ]:
def fit_catboost(train, tune, columns, seed):
    from catboost import CatBoostClassifier

    # Cat18 and Cat28 change only their selected input columns.
    model = CatBoostClassifier(
        iterations=TREE_ITERATIONS, depth=6, learning_rate=0.05,
        loss_function="Logloss", eval_metric="PRAUC", auto_class_weights="SqrtBalanced",
        random_seed=seed, thread_count=THREADS,
        allow_writing_files=False, verbose=False,
    )
    # Only tune controls early stopping; test is never passed to fit().
    model.fit(
        train[columns], train["target"],
        eval_set=(tune[columns], tune["target"]),
        # Example: 80 unimproved rounds can stop before the 1,000-round cap.
        early_stopping_rounds=TREE_PATIENCE,
    )
    return model

In [ ]:
def fit_xgboost(train, tune, columns, seed):
    from xgboost import XGBClassifier

    positive = train["target"].sum()
    # Example: 75 normal / 25 abnormal -> sqrt(75/25) = sqrt(3).
    class_weight = np.sqrt((len(train) - positive) / positive)
    # XGBoost uses depth 4 and the same iteration/patience budget.
    model = XGBClassifier(
        n_estimators=TREE_ITERATIONS, max_depth=4, learning_rate=0.05,
        eval_metric="aucpr", tree_method="hist", n_jobs=THREADS,
        random_state=seed, scale_pos_weight=class_weight,
        early_stopping_rounds=TREE_PATIENCE,
    )
    model.fit(
        train[columns], train["target"],
        eval_set=[(tune[columns], tune["target"])], verbose=False,
    )
    return model

In [ ]:
# A tiny input shows what the additive transform does before real training.
example_features = pd.DataFrame({
    "varying": [110.0, 130.0, np.nan, 170.0],
    "constant": [140.0, 140.0, 140.0, 140.0],
})
example_labels = np.array([0, 0, 1, 1])
example_model = make_logistic_model(spline=True).fit(example_features, example_labels)
print("input shape =", example_features.shape)
print("imputation medians =", example_model.named_steps["simpleimputer"].statistics_)
# One varying column gives 6 basis values; the constant column is removed.
print("transformed shape =", example_model[:-1].transform(example_features).shape)
print("probabilities =", example_model.predict_proba(example_features)[:, 1])

## 3. 후보 11개를 학습하고 검증 AP로 고른다

트리 세 계열은 각각 seed 세 개를 모두 실행한다. 선형·spline 가산 모델은 seed 42 하나다. **한 계열의 예정된 seed가 하나라도 실패하면 그 계열의 선택을 보류**한다. 점수가 같으면 작은 seed를 고른다. 검증 AP는 선택에만 쓰며 시험 점수는 선택을 바꾸지 않는다.

In [ ]:
def make_prediction(frame, scores):
    # Example: row_id="DEMO221841_0:0", score=0.7 means P(abnormal)=0.7.
    prediction = frame[PREDICTION_COLUMNS[:-2]].copy()
    # Scores are row-aligned: scores[i] belongs to frame.iloc[i].
    prediction["score"] = np.asarray(scores, dtype=float)
    prediction["threshold"] = np.nan  # No alarm threshold is selected here.
    if prediction["row_id"].duplicated().any():
        raise ValueError("예측 구간 ID가 중복됩니다")
    if not np.isfinite(prediction["score"]).all():
        raise ValueError("예측 확률에 비유한 값이 있습니다")
    return prediction

In [ ]:
def fit_candidate(arm, columns, seed):
    start = time.perf_counter()
    if arm in ("cat28", "cat18"):
        model = fit_catboost(training, validation, columns, seed)
        history = model.get_evals_result()
        # rounds counts evaluations; best_iteration is the chosen tree count.
        rounds = len(history["validation"]["PRAUC"])
        best_iteration = model.get_best_iteration() + 1
    elif arm == "xgb28":
        model = fit_xgboost(training, validation, columns, seed)
        rounds = len(model.evals_result()["validation_0"]["aucpr"])
        best_iteration = model.best_iteration + 1
    else:
        model = make_logistic_model(spline=arm == "additive28")
        # The additive arms use only train, without early stopping on val.
        model.fit(training[columns], training["target"])
        rounds, best_iteration = 0, 0
    fit_seconds = time.perf_counter() - start

    # Save validation and test probabilities, but score selection on val only.
    evaluation_pool = pd.concat([validation, testing])
    probabilities = model.predict_proba(evaluation_pool[columns])[:, 1]
    prediction = make_prediction(evaluation_pool, probabilities)
    val_rows = prediction.loc[prediction["split"].eq("val")]
    # Example: a 0.60 validation AP ranks this seed; test AP cannot select it.
    val_ap = average_precision_score(val_rows["target"], val_rows["score"])

    # Three identical test predictions estimate inference time per window.
    start = time.perf_counter()
    for _ in range(3):
        model.predict_proba(testing[columns])
    # Divide total microseconds by 3 runs × number of test windows.
    # Example: 0.015 s / (3 × 5 windows) = 1,000 µs per window.
    microseconds = (time.perf_counter() - start) * 1e6 / (3 * len(testing))
    diagnostic = dict(
        model=arm, seed=seed, fit_seconds=fit_seconds,
        best_iteration=best_iteration, iterations=rounds,
        hit_cap=rounds == TREE_ITERATIONS,
        prediction_us_per_window=microseconds,
    )
    return model, prediction, float(val_ap), diagnostic

In [ ]:
# The seed tie rule can be checked without training 11 models.
example_candidates = [(0.50, 42), (0.52, 43), (0.52, 44)]
chosen_ap, chosen_seed = max(example_candidates, key=lambda item: (item[0], -item[1]))
print("validation AP and seed =", chosen_ap, chosen_seed)
# Expected: 0.52 and 43; seed 44 does not win a tie.

In [ ]:
models = {}
seed_predictions = {}
selection_rows = []
training_rows = []

if ready:
    for arm, columns in ARMS.items():
        planned_seeds = SEEDS if arm in ("cat28", "cat18", "xgb28") else (42,)
        for seed in planned_seeds:
            try:
                model, prediction, val_ap, diagnostic = fit_candidate(arm, columns, seed)
                models[(arm, seed)] = model
                seed_predictions[(arm, seed)] = prediction
                # One ranking row per successful arm/seed, even before selection.
                selection_rows.append(dict(model=arm, seed=seed, val_AP=val_ap))
                training_rows.append(diagnostic)
            except Exception as error:
                # Example: if cat28 seed 43 fails, still run seed 44.
                issues.append(dict(stage="fit", job=f"{arm}:{seed}", reason=str(error)))
        print(arm, "successful candidates =", sum(row["model"] == arm for row in selection_rows))

In [ ]:
selection_table = pd.DataFrame(selection_rows, columns=["model", "seed", "val_AP"])
# fit_seconds and prediction_us_per_window are timings, not model scores.
training_table = pd.DataFrame(training_rows, columns=[
    "model", "seed", "fit_seconds", "best_iteration", "iterations",
    "hit_cap", "prediction_us_per_window",
])
print("successful candidates =", len(selection_table), "/ 11")
display(selection_table)
display(training_table)

In [ ]:
selected = {}
selected_predictions = {}
if ready:
    for arm in ARMS:
        planned_seeds = SEEDS if arm in ("cat28", "cat18", "xgb28") else (42,)
        candidates = selection_table.loc[selection_table["model"].eq(arm)]
        if len(candidates) != len(planned_seeds):
            # Do not select a tree family from only two of its three seeds.
            issues.append(dict(stage="selection", job=arm, reason="사전 seed 집합 미완료: 선택 보류"))
            continue

        # Example: AP 0.52 at seeds 43 and 44 -> choose seed 43.
        best = max(candidates.itertuples(), key=lambda row: (row.val_AP, -row.seed))
        selected[arm] = dict(seed=int(best.seed), val_AP=float(best.val_AP))
        # Keep this already-fitted model; do not refit on train + val.
        selected_predictions[arm] = seed_predictions[(arm, best.seed)]
print("selected =", selected)

In [ ]:
if "cat28" in selected_predictions:
    print("Cat28 prediction rows =", len(selected_predictions["cat28"]))

## 4. 시험 산모를 함께 재표집해 성능과 차이를 구한다

AP·AUROC·Brier를 같은 시험 구간에서 계산한다. 산모 한 명을 두 번 뽑으면 그 산모의 모든 구간을 두 번 센다. 두 모델의 차이는 **같은 산모 추출 결과**를 양쪽에 적용한 뒤 `Cat28 − 비교 모델`로 구한다. 이 95% 구간은 이미 고정한 모델의 시험 예측에 대한 것이며 재학습 변동은 포함하지 않는다.

In [ ]:
def align_predictions(left, right):
    # Example: left rows [A, B], right rows [B, A] -> reorder right to [A, B].
    # A missing row is an error; do not silently compare an intersection.
    if left["row_id"].duplicated().any() or right["row_id"].duplicated().any():
        raise ValueError("비교 예측의 구간 ID가 중복됩니다")
    if set(left["row_id"]) != set(right["row_id"]):
        raise ValueError("두 모델의 평가 구간이 다릅니다")
    # Align by segment ID before subtracting AP/AUROC/Brier.
    aligned = right.set_index("row_id").loc[left["row_id"]].reset_index()
    for column in ("mother_id", "target", "record_id"):
        if not np.array_equal(left[column].to_numpy(), aligned[column].to_numpy()):
            raise ValueError("예측 정렬 불일치: " + column)
    return aligned

In [ ]:
def prepare_metrics(prediction):
    labels = prediction["target"].to_numpy(dtype=int)
    scores = prediction["score"].to_numpy(dtype=float)
    if not len(prediction) or set(labels) - {0, 1} or not np.isfinite(scores).all():
        raise ValueError("평가 라벨·점수가 올바르지 않거나 평가 집합이 비었습니다")

    # Example: mother IDs [A, A, B] -> mother_index [0, 0, 1].
    mothers, mother_index = np.unique(prediction["mother_id"], return_inverse=True)
    # Example: scores [0.2, 0.9, 0.5] -> descending row order [1, 2, 0].
    order = np.argsort(-scores, kind="stable")
    # Example: sorted scores [0.9, 0.9, 0.3] -> group ends [1, 2].
    # Tied scores are one threshold, independent of their row order.
    ends = np.r_[np.flatnonzero(np.diff(scores[order])), len(prediction) - 1]
    return dict(
        labels=labels, scores=scores, mothers=mothers,
        mother_index=mother_index, order=order, ends=ends,
    )

In [ ]:
def weighted_metrics(data, counts=None):
    if counts is None:
        counts = np.ones(len(data["mothers"]))
    # Example: mother counts [2, 0] -> weights [2, 2, 0] for [A, A, B].
    weights = counts[data["mother_index"]]
    labels = data["labels"]
    # Denominators count weighted abnormal and normal *windows*.
    positive = np.sum(weights * labels)
    negative = np.sum(weights * (1 - labels))
    if not positive or not negative:
        return None

    order, ends = data["order"], data["ends"]
    true_positive = np.r_[0, np.cumsum((weights * labels)[order])[ends]]
    false_positive = np.r_[0, np.cumsum((weights * (1 - labels))[order])[ends]]
    recall = true_positive / positive
    false_positive_rate = false_positive / negative
    predicted_positive = np.maximum(true_positive[1:] + false_positive[1:], 1e-30)
    # Example: descending labels [1, 0, 1, 0] -> AP = (1 + 2/3)/2 = 5/6.
    ap = np.sum(np.diff(recall) * true_positive[1:] / predicted_positive)
    # Those labels with distinct scores give AUROC=3/4.
    auc = np.sum(np.diff(false_positive_rate) * (recall[1:] + recall[:-1]) / 2)
    # Example: label 1, score 0.9 contributes (0.9 - 1)^2 = 0.01.
    brier = np.sum(weights * (data["scores"] - labels) ** 2) / weights.sum()
    return dict(AP=ap, AUROC=auc, Brier=brier)

In [ ]:
# Tiny comparison: same rows and same mother counts for both arms.
example_left = pd.DataFrame({
    "row_id": ["a", "b", "c", "d"],
    "record_id": ["r1", "r1", "r2", "r2"],
    "mother_id": ["A", "A", "B", "B"],
    "target": [1, 0, 1, 0],
    "score": [0.9, 0.8, 0.7, 0.1],
})
example_right = example_left.copy()
example_right["score"] = [0.8, 0.9, 0.7, 0.1]
left_data = prepare_metrics(example_left)
right_data = prepare_metrics(align_predictions(example_left, example_right))
print("left AP, AUROC =", weighted_metrics(left_data)["AP"], weighted_metrics(left_data)["AUROC"])
print("right AP, AUROC =", weighted_metrics(right_data)["AP"], weighted_metrics(right_data)["AUROC"])
example_counts = np.array([2, 0])
print("same row weights =", example_counts[left_data["mother_index"]].tolist())

In [ ]:
def score_intervals(left, right=None, n_boot=N_BOOTSTRAP, seed=BOOTSTRAP_SEED):
    left_data = prepare_metrics(left)
    right_data = prepare_metrics(align_predictions(left, right)) if right is not None else None
    left_point = weighted_metrics(left_data)
    right_point = weighted_metrics(right_data) if right_data is not None else None
    if left_point is None or (right_data is not None and right_point is None):
        raise ValueError("시험 평가에는 정상·비정상 구간이 모두 필요합니다")
    # Example: left AP 0.60, right AP 0.55 -> paired AP point 0.05.
    point = {
        metric: value - right_point[metric] if right_data is not None else value
        for metric, value in left_point.items()
    }
    samples = {metric: [] for metric in point}
    rng = np.random.default_rng(seed)
    mother_count = len(left_data["mothers"])

    for _ in range(n_boot):
        # Example: draw [0, 0, 2] among 3 mothers -> counts [2, 0, 1].
        counts = np.bincount(rng.integers(mother_count, size=mother_count), minlength=mother_count)
        # The same mother counts go to both arms, preserving the pairing.
        left_result = weighted_metrics(left_data, counts)
        right_result = weighted_metrics(right_data, counts) if right_data is not None else None
        if left_result is None or (right_data is not None and right_result is None):
            continue
        for metric in point:
            value = left_result[metric] - right_result[metric] if right_data is not None else left_result[metric]
            if np.isfinite(value):
                samples[metric].append(value)

    rows = []
    for metric, value in point.items():
        low, high = np.nan, np.nan
        if samples[metric]:
            # Use 2.5th and 97.5th percentiles of valid mother draws.
            low, high = np.quantile(samples[metric], [0.025, 0.975])
        rows.append(dict(
            metric=metric, point=value, low=low, high=high,
            # Invalid one-class draws are omitted from this count.
            valid_bootstrap=len(samples[metric]),
            mothers=mother_count, segments=len(left),
        ))
    return pd.DataFrame(rows)

In [ ]:
# For an identical prediction table, every paired point and interval is zero.
example_delta = score_intervals(example_left, example_left, n_boot=20)
print("identical-model differences =")
display(example_delta[["metric", "point", "low", "high"]])

In [ ]:
# 03's single-model predictions are optional comparators, not training inputs.
test_predictions = {}
for arm, prediction in selected_predictions.items():
    test_predictions[arm] = prediction.loc[prediction["split"].eq("test")].copy()

if ready:
    for arm in ("single_linear", "single_spline"):
        path = RESULTS / f"pred_{arm}.csv"
        try:
            prediction = pd.read_csv(
                path, dtype={key: str for key in ("row_id", "record_id", "mother_id", "site")},
            )
            # 03 saved val + test; only test rows enter this comparison.
            test_predictions[arm] = prediction.loc[prediction["split"].eq("test")].copy()
        except (OSError, ValueError, KeyError) as error:
            issues.append(dict(stage="comparison", job=arm, reason=str(error)))

print("test rows by arm =", {arm: len(table) for arm, table in test_predictions.items()})

In [ ]:
metric_rows = []
paired_rows = []
if "cat28" in test_predictions:
    # Cat28 is the left side of every H1/H2 paired difference.
    base = test_predictions["cat28"]
    for arm, prediction in test_predictions.items():
        try:
            metric_rows.append(score_intervals(prediction).assign(model=arm))
            if arm != "cat28":
                paired_rows.append(score_intervals(base, prediction).assign(comparison="cat28 - " + arm))
        except ValueError as error:
            issues.append(dict(stage="test", job=arm, reason=str(error)))

In [ ]:
METRIC_COLUMNS = [
    "metric", "point", "low", "high", "valid_bootstrap", "mothers", "segments", "model",
]
PAIRED_COLUMNS = METRIC_COLUMNS[:-1] + ["comparison"]
test_metrics = pd.concat(metric_rows, ignore_index=True) if metric_rows else pd.DataFrame(columns=METRIC_COLUMNS)
paired_table = pd.concat(paired_rows, ignore_index=True) if paired_rows else pd.DataFrame(columns=PAIRED_COLUMNS)
# Example: 7 available arms × 3 metrics -> 21 test metric rows.
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(test_metrics)
    display(paired_table)


In [ ]:
if "cat28" in test_predictions:
    figure, axis = plt.subplots(figsize=(7, 5))
    for arm, prediction in test_predictions.items():
        if prediction.empty:
            continue
        PrecisionRecallDisplay.from_predictions(
            prediction["target"], prediction["score"], name=arm, ax=axis,
        )
    # A no-skill AP reference is the abnormal-window fraction in this test split.
    axis.axhline(testing["target"].mean(), linestyle="--", color="gray", label="prevalence")
    axis.legend(fontsize=8)
    plt.show()

In [ ]:
def show_differences(table, metric, title):
    selected = table.loc[table["metric"].eq(metric)]
    if selected.empty:
        print("No comparison rows for", metric)
        return
    figure, axis = plt.subplots(figsize=(8, max(3, len(selected) * 0.4)))
    positions = np.arange(len(selected))
    # A line crossing zero means this bootstrap interval includes no difference.
    axis.hlines(positions, selected["low"], selected["high"], color="C0")
    axis.scatter(selected["point"], positions)
    axis.axvline(0, color="gray", linestyle="--")
    axis.set_yticks(positions, selected["comparison"])
    axis.set_xlabel(metric + " difference (Cat28 minus comparator)")
    axis.set_title(title)
    figure.tight_layout()
    plt.show()

In [ ]:
show_differences(paired_table, "AUROC", "H2: Cat28 minus comparators")
show_differences(paired_table, "AP", "H1: Cat28 minus comparators")

In [ ]:
# Matched-seed differences are point estimates: no bootstrap interval here.
same_seed_rows = []
for seed in SEEDS:
    left = seed_predictions.get(("cat28", seed))
    if left is None:
        continue
    # Example: seed 43 Cat28 is compared with seed 43 Cat18/XGB28.
    left_test = left.loc[left["split"].eq("test")]
    for arm in ("cat18", "xgb28"):
        right = seed_predictions.get((arm, seed))
        if right is None:
            continue
        right_test = right.loc[right["split"].eq("test")]
        try:
            delta = score_intervals(left_test, right_test, n_boot=0)
            same_seed_rows.append(delta.assign(seed=seed, comparison="cat28 - " + arm))
        except ValueError as error:
            issues.append(dict(stage="same_seed", job=f"{arm}:{seed}", reason=str(error)))

SAME_SEED_COLUMNS = PAIRED_COLUMNS[:-1] + ["seed", "comparison"]
same_seed_table = (
    pd.concat(same_seed_rows, ignore_index=True)
    if same_seed_rows else pd.DataFrame(columns=SAME_SEED_COLUMNS)
)
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(same_seed_table)


## 5. 봉인된 test 없이 개발 집합에서 반복한다

`train ∪ val` 안에서 산모 단위 5-fold를 만든다. 매 fold의 나머지 산모에서 다시 inner train과 inner validation을 나눈다. 단일 spline의 28개 특징 선택과 CatBoost 조기중단은 inner validation만 사용한다. 세 반복 seed마다 외부 fold와 내부 분할을 다시 만든다. 같은 반복의 세 모델은 같은 held-out 산모를 예측한다.

OOF(out-of-fold) 예측은 각 산모가 그때 학습에 없었을 때의 확률이다. 이는 개발 코호트 내부의 탐색적 반복성 점검이며 봉인된 test의 대체 결과가 아니다.

In [ ]:
def split_inner_mothers(pool, fraction, seed):
    # Stratify mothers by whether any of their windows is abnormal.
    # Example: one mother with labels [0, 1] gets mother label 1.
    # Split IDs, not windows, so twins and repeated recordings stay together.
    mother_labels = pool.groupby("mother_id")["target"].max()
    inner_ids, tune_ids = train_test_split(
        mother_labels.index.to_numpy(), test_size=fraction,
        stratify=mother_labels.to_numpy(), random_state=seed,
    )
    inner = pool.loc[pool["mother_id"].isin(inner_ids)].copy()
    tune = pool.loc[pool["mother_id"].isin(tune_ids)].copy()
    return inner, tune

In [ ]:
def outer_folds(development, seed):
    # Example: 5 folds -> each mother is held out once per repetition.
    splitter = StratifiedGroupKFold(CV_FOLDS, shuffle=True, random_state=seed)
    for pool_indices, held_indices in splitter.split(
        development, development["target"], development["mother_id"],
    ):
        # Each held-out mother is absent from this fold's training pool.
        yield development.iloc[pool_indices].copy(), development.iloc[held_indices].copy()

In [ ]:
def fit_cv_fold(inner, tune, held, seed):
    outputs = {}
    # A held-out fold is used only for prediction, never early stopping.
    tree = fit_catboost(inner, tune, CAT28, seed)
    outputs["cat28"] = make_prediction(held, tree.predict_proba(held[CAT28])[:, 1])

    # Select one of 28 single features using only this fold's inner validation.
    options = []
    for feature in CAT28:
        model = make_logistic_model(spline=True).fit(inner[[feature]], inner["target"])
        scores = model.predict_proba(tune[[feature]])[:, 1]
        # The *inner* AUROC selects this fold's one spline feature.
        auc = roc_auc_score(tune["target"], scores)
        options.append((auc, feature, model))
    # A tie keeps the first feature in CAT28 order.
    best_auc, best_feature, single = max(options, key=lambda item: item[0])
    single_scores = single.predict_proba(held[[best_feature]])[:, 1]
    outputs["single_spline"] = make_prediction(held, single_scores)

    additive = make_logistic_model(spline=True).fit(inner[CAT28], inner["target"])
    # All three arms predict the identical held-out rows in this fold.
    outputs["additive28"] = make_prediction(held, additive.predict_proba(held[CAT28])[:, 1])
    choice = dict(single_feature=best_feature, inner_val_AUROC=best_auc)
    return outputs, choice

In [ ]:
# Inspect one real split before fitting the full CV experiment.
if ready:
    preview_development = feature_table.loc[feature_table["split"].ne("test")]
    preview_pool, preview_held = next(outer_folds(preview_development, SEEDS[0]))
    preview_inner, preview_tune = split_inner_mothers(preview_pool, 0.2, SEEDS[0])
    print("inner train mothers =", preview_inner["mother_id"].nunique())
    print("inner validation mothers =", preview_tune["mother_id"].nunique())
    print("outer held-out mothers =", preview_held["mother_id"].nunique())
    # Expected: no held-out mother can appear in this fold's fitting data.
    overlap = set(preview_pool["mother_id"]) & set(preview_held["mother_id"])
    print("pool / held-out mother overlap =", len(overlap))

In [ ]:
cv_predictions = {}
cv_choice_rows = []
if ready:
    # Development = original train + val; the saved test rows stay sealed.
    development = feature_table.loc[feature_table["split"].ne("test")].copy()
    for seed in SEEDS:
        fold_outputs = {arm: [] for arm in ("cat28", "single_spline", "additive28")}
        for fold, (pool, held) in enumerate(outer_folds(development, seed)):
            try:
                inner, tune = split_inner_mothers(pool, 0.2, seed + fold)
                if any(set(part["target"].unique()) != {0, 1} for part in (inner, tune, held)):
                    raise ValueError("CV fold 라벨 부족")
                outputs, choice = fit_cv_fold(inner, tune, held, seed)
                for arm, prediction in outputs.items():
                    fold_outputs[arm].append(prediction.assign(fold=fold))
                # Example: seed 42, fold 0 may select hist_width; fold 1 may differ.
                cv_choice_rows.append(dict(seed=seed, fold=fold, **choice))
            except Exception as error:
                issues.append(dict(stage="cv_fold", job=f"{seed}:{fold}", reason=str(error)))
        # Incomplete repetitions are not reported as full OOF estimates.
        # A 4/5-fold repetition is incomplete and gets no OOF metric.
        if all(len(parts) == CV_FOLDS for parts in fold_outputs.values()):
            repetition = {}
            for arm, parts in fold_outputs.items():
                oof = pd.concat(parts, ignore_index=True)
                if set(oof["row_id"]) != set(development["row_id"]) or oof["row_id"].duplicated().any():
                    issues.append(dict(stage="cv_coverage", job=f"{arm}:{seed}", reason="CV coverage 오류"))
                    break
                repetition[(arm, seed)] = oof
            if len(repetition) == len(fold_outputs):
                cv_predictions.update(repetition)
        print("CV seed", seed, "completed folds =", len(fold_outputs["cat28"]))

In [ ]:
# A choice row names the feature selected inside one inner validation split.
print("CV feature choices =", len(cv_choice_rows))
display(pd.DataFrame(cv_choice_rows).head(3))
if cv_predictions:
    sample_key = next(iter(cv_predictions))
    sample_oof = cv_predictions[sample_key]
    print("OOF arm and seed =", sample_key)
    print("OOF rows =", len(sample_oof))

In [ ]:
cv_metric_rows = []
cv_paired_rows = []
for seed in SEEDS:
    arms = ("cat28", "single_spline", "additive28")
    if any((arm, seed) not in cv_predictions for arm in arms):
        continue
    for arm in arms:
        oof = cv_predictions[(arm, seed)]
        # OOF point scores summarize held-out development predictions.
        cv_metric_rows.append(score_intervals(oof, n_boot=0).assign(model=arm, seed=seed))
    for arm in ("single_spline", "additive28"):
        delta = score_intervals(cv_predictions[("cat28", seed)], cv_predictions[(arm, seed)])
        # Bootstrap resamples the shared development mothers for this seed.
        cv_paired_rows.append(delta.assign(comparison="cat28 - " + arm, seed=seed))

In [ ]:
CV_METRIC_COLUMNS = METRIC_COLUMNS + ["seed"]
CV_PAIRED_COLUMNS = PAIRED_COLUMNS + ["seed"]
CV_CHOICE_COLUMNS = ["seed", "fold", "single_feature", "inner_val_AUROC"]
cv_metrics = (
    pd.concat(cv_metric_rows, ignore_index=True)
    if cv_metric_rows else pd.DataFrame(columns=CV_METRIC_COLUMNS)
)
cv_paired = (
    pd.concat(cv_paired_rows, ignore_index=True)
    if cv_paired_rows else pd.DataFrame(columns=CV_PAIRED_COLUMNS)
)
cv_choices = pd.DataFrame(cv_choice_rows, columns=CV_CHOICE_COLUMNS)


In [ ]:
# OOF scores and paired differences describe held-out development predictions.
# Example: point=0.03, low=0.01, high=0.05 means an estimated +0.03 AP difference.
# Model/seed/fold summaries contain no individual IDs or prediction rows.
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(cv_metrics)
    display(cv_paired)
    display(cv_choices)

In [ ]:
if not cv_metrics.empty:
    # One bar per model and repetition; these are development OOF scores.
    ap_rows = cv_metrics.loc[cv_metrics["metric"].eq("AP")]
    ap_rows.pivot(index="seed", columns="model", values="point").plot.bar(
        title="Development OOF AP by fold / training seed",
    )
    plt.show()

## 6. 후보·선택·평가를 저장한다

`seeds/pred_*`에는 성공한 개별 seed의 검증·시험 예측을, 루트의 `pred_*`에는 해당 계열에서 선택한 seed의 예측을 둔다. CatBoost 객체는 `models/*.cbm`에 저장한다. 그 결과는 이후 지표 기여 실험의 기준이 된다. 시험 표는 모델별 절대 성능, 같은 시험 산모의 차이, 같은 seed끼리의 차이를 따로 담는다. 실패 또는 분할 미준비는 `04_issues.csv`와 `04_status.json`에서 확인한다.

In [ ]:
RESULTS.mkdir(parents=True, exist_ok=True)
(RESULTS / "seeds").mkdir(exist_ok=True)
(RESULTS / "models").mkdir(exist_ok=True)

selection_table.to_csv(RESULTS / "04_selection.csv", index=False)
training_table.to_csv(RESULTS / "04_training.csv", index=False)
(RESULTS / "model_selection.json").write_text(
    json.dumps(selected, ensure_ascii=False, indent=2), encoding="utf-8",
)

# Rewrite every planned path so a failed rerun cannot expose an old prediction.
for arm in ARMS:
    planned_seeds = SEEDS if arm in ("cat28", "cat18", "xgb28") else (42,)
    for seed in planned_seeds:
        prediction = seed_predictions.get((arm, seed), pd.DataFrame(columns=PREDICTION_COLUMNS))
        prediction.to_csv(RESULTS / "seeds" / f"pred_{arm}_{seed}.csv", index=False)
        if arm in ("cat28", "cat18"):
            model_path = RESULTS / "models" / f"{arm}_{seed}.cbm"
            if (arm, seed) in models:
                models[(arm, seed)].save_model(str(model_path))
            else:
                model_path.unlink(missing_ok=True)
    # An unselected arm gets a header-only file instead of an old success.
    prediction = selected_predictions.get(arm, pd.DataFrame(columns=PREDICTION_COLUMNS))
    prediction.to_csv(RESULTS / f"pred_{arm}.csv", index=False)

In [ ]:
# Test comparisons use validation-selected models; CV uses OOF predictions.
test_metrics.to_csv(RESULTS / "04_test_metrics.csv", index=False)
paired_table.to_csv(RESULTS / "04_paired.csv", index=False)
same_seed_table.to_csv(RESULTS / "04_same_seed.csv", index=False)
cv_metrics.to_csv(RESULTS / "04_cv_metrics.csv", index=False)
cv_paired.to_csv(RESULTS / "04_cv_paired.csv", index=False)
cv_choices.to_csv(RESULTS / "04_cv_choices.csv", index=False)

for seed in SEEDS:
    for arm in ("cat28", "single_spline", "additive28"):
        empty_oof = pd.DataFrame(columns=PREDICTION_COLUMNS + ["fold"])
        oof = cv_predictions.get((arm, seed), empty_oof)
        oof.to_csv(RESULTS / f"cv_{arm}_{seed}.csv", index=False)

In [ ]:
issue_table = pd.DataFrame(issues, columns=["stage", "job", "reason"])
issue_table.to_csv(RESULTS / "04_issues.csv", index=False)
 # Count complete repetitions, not just successfully trained folds.
completed_cv_seeds = sum(("cat28", seed) in cv_predictions for seed in SEEDS)
if not ready:
    status_name = "unavailable"
elif len(selected) == len(ARMS) and completed_cv_seeds == len(SEEDS) and issue_table.empty:
    status_name = "complete"
else:
    status_name = "partial"
status = dict(
    status=status_name, selected_models=selected,
    completed_cv_seeds=completed_cv_seeds, expected_cv_seeds=len(SEEDS),
    input_error=input_error, bootstrap=N_BOOTSTRAP, bootstrap_seed=BOOTSTRAP_SEED,
)
(RESULTS / "04_status.json").write_text(
    json.dumps(status, ensure_ascii=False, indent=2), encoding="utf-8",
)
print("status =", status_name)
print("selected models =", len(selected), "/", len(ARMS))
print("complete CV seeds =", completed_cv_seeds, "/", len(SEEDS))
print("issues =", len(issue_table))
if not issue_table.empty:
    print("issue count =", len(issue_table))

## 결과를 읽을 때

Cat28과 단일 spline의 차이는 이 두 **제한된 모델군** 사이의 차이다. Cat28과 28특징 spline 가산 모델의 차이도 특정 상호작용의 인과 효과를 증명하지 않는다. 신뢰구간은 고정된 선택 모델·현재 시험 산모에 대한 것이며 학습과 seed 선택의 변동은 담지 않는다. 반복 개발 CV와 holdout은 같은 코호트에서 나왔으므로 독립 연구 여러 개로 세지 않는다. 과거 연구에서 노출된 산모 여부를 알 수 없다는 01의 한계도 그대로 따른다.

### 반출용 제외·오류 집계

개별 ID·경로·오류 전문을 제외한 분류별 건수다.

In [ ]:
show_issue_counts(issue_table["reason"])